In [1]:
%load_ext autoreload
%autoreload 2

import os,sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import util as yu
from util import *
import util_Nsgm as yu2

yu.setpath('plot_paper')

ens='b'
tfs=[8,10,12,14,16,18,20]
yunit_mul=yu.ens2amul_iso[ens]*yu.ens2aInv[ens]

mpl.rcParams['lines.markersize'] = mpl.rcParams['errorbar.capsize'] = 7
# mpl.rcParams['axes.labelsize'] = mpl.rcParams['axes.titlesize'] = mpl.rcParams['xtick.labelsize'] = mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['font.size'] = 20
# yu.mpl_global_elinewidth=yu.mpl_global_capthick=3

In [2]:
[c2ptM,tf2c3ptM,c2ptCorrDic_NJN]=yu.load_pkl_reg('data',pathlabel='processData')

In [3]:
# 2pt GEVP

dt=2
t0s=np.arange(0,22-dt)
ts=t0s+dt

t=[yu.GEVP(c,t0s,tList=ts) for c in c2ptM]
evals=np.array([eval for eval,evec in t])
evecs=np.array([evec for eval,evec in t])
evecsInv=np.linalg.inv(evecs)
# print(evals.shape,evecs.shape)

fig,axs=yu.getFigAxs(2,2,sharex='col',sharey='row',Lrow=3,Lcol=6)
xunit=yu.ens2a[ens]

xmin_plt=1; xmax_plt=16
ax=axs[0,0]; yunit=1; color='r'
xmin_select=7; xmax_select=12
ax.set_ylim([-0.02,0.05])
t=np.real(evecs[:,:,0,1])/np.real(evecs[:,:,0,0])
mean,err=yu.jackme(t)
xmin=1
plt_x=ts[xmin_plt:xmax_plt]*xunit; plt_y=mean[xmin_plt:xmax_plt]*yunit; plt_yerr=err[xmin_plt:xmax_plt]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
plt_x=ts[xmin_select:xmax_select]*xunit; plt_y=mean[xmin_select:xmax_select]*yunit; plt_yerr=err[xmin_select:xmax_select]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white')

ax=axs[0,1]
xmins=np.arange(1,xmax_select-1)
fits=yu.doFits_const(t,xmins,[xmax_select],corrQ=False)
for fit in fits:
    (xmin,xmax),pars_jk,chi2_jk,Ndof = fit
    mean,err=yu.jackme(pars_jk)
    plt_x=ts[xmin]*xunit; plt_y=mean[0]*yunit; plt_yerr=err[0]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white' if xmin==xmin_select else None)
    
    if xmin==xmin_select:
        v_selected=pars_jk[:,0]
        ax.axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y,plt_yerr))
        axs[0,0].axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2)
# ax.legend()

ax=axs[1,0]; yunit=1
# ax.set_ylim([-0.01,0.01])
t=1/(np.real(evecs[:,:,0,0])*np.real(evecsInv[:,:,0,0]))-1
mean,err=yu.jackme(t)
xmin=1
plt_x=ts[xmin_plt:xmax_plt]*xunit; plt_y=mean[xmin_plt:xmax_plt]*yunit; plt_yerr=err[xmin_plt:xmax_plt]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
plt_x=ts[xmin_select:xmax_select]*xunit; plt_y=mean[xmin_select:xmax_select]*yunit; plt_yerr=err[xmin_select:xmax_select]*yunit
ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white')

ax=axs[1,1]
xmins=np.arange(1,xmax_select-1)
fits=yu.doFits_const(t,xmins,[xmax_select],corrQ=False)
for fit in fits:
    (xmin,xmax),pars_jk,chi2_jk,Ndof = fit
    mean,err=yu.jackme(pars_jk)
    plt_x=ts[xmin]*xunit; plt_y=mean[0]*yunit; plt_yerr=err[0]*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,mfc='white' if xmin==xmin_select else None)
    
    if xmin==xmin_select:
        w_selected=pars_jk[:,0]
        ax.axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y,plt_yerr))
        axs[1,0].axhspan(plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2)
# ax.legend()

axs[0,0].set_ylabel(r'$v_{0,N\sigma}/v_{0,N}$')
axs[1,0].set_ylabel(r'W')

axs[1,0].set_xlabel(r'$t$ [fm]')
axs[1,1].set_xlabel(r'$t_{\rm low}$ [fm]')

axs[0,0].set_yticks([-0.02,0,0.02,0.04])
axs[0,0].set_ylim([-0.025,0.045])
axs[1,0].set_yticks([0,0.003,0.006,0.009])

axs[1,0].set_xticks(np.arange(0,1.6,0.2))
axs[1,0].set_xlim([0,1.5])
axs[1,1].set_xticks(np.arange(0,1.2,0.2))
axs[1,1].set_xlim([0,1.1])

yu.finalizePlot('GEVP_vw')

In [4]:
[c2ptM,tf2c3ptM,c2ptCorrDic_NJN]=yu.load_pkl_reg('data',pathlabel='processData')
tf2c3ptM={tf:np.real(tf2c3ptM[tf]) for tf in tf2c3ptM.keys()}

t=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_2pt')
[pars_jk_meff1st,pars_jk_meff2st,pars_jk_meff3st]=[t[0][ens],t[1][ens],t[2][ens]]
[v_selected,w_selected]=yu.load_pkl_reg('vw',pathlabel='analysis_2ptGEVP')

In [5]:
tf2ratio_noGEVP={tf:tf2c3ptM[tf][:,:,0,0]/c2ptCorrDic_NJN[tf][:,None] for tf in tfs}
tf2ratio_GEVP={}
for tf in tfs:
    t=[ (c3ptM[:,0,0]*(1-w**2) + v*(1+w)*(c3ptM[:,0,1]+c3ptM[:,1,0]) ) /  \
        (c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) + v**2*c2ptM[tf,1,1] )[None]    \
        for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)]
    tf2ratio_GEVP[tf]=np.array(t)

In [6]:
dic={'tf2ratio':tf2ratio_noGEVP, 'mfc:[global]':['white'], 'shift:[rainbow,midpoint,fit]':[0.2,0.2,0], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'])
dic={'tf2ratio':tf2ratio_GEVP, 'mfc:[global]':['None'], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'],figAxs=(fig,axs))
# fig.suptitle('open (R_std); filled (R_GEVP)')

ax=axs[0,0]
ax.set_ylabel(r'$\sigma_{\pi N}$ [MeV]')
ax.set_ylim([0,80])
ax.set_yticks([0,20,40,60,80])
ax.set_xlim([-0.8,0.8])
ax.set_xticks(np.arange(-0.6,0.7,0.2))
handles=[plt.errorbar(np.nan,np.nan,yerr=np.nan,color=color,fmt=fmt,mfc=mfc) for color,fmt,mfc in zip(['black','black'],['s','s'],['white',None])]
yu.legend(ax,labels=[r'$R_{\rm std}$',r'$R_{\rm GEVP}$'],ncols=2,tightQ=True,handles=handles,fontsize=20)

ax=axs[0,1]
ax.set_xticks(np.arange(0.7,1.6,0.3))

yu.finalizePlot('Rstd_RGEVP')

In [7]:
tf2ratio_GEVP_w={}
for tf in tfs:
    t=[ (c3ptM[:,0,0]*(1-w**2) + v*(1+w)*(c3ptM[:,0,1]+c3ptM[:,1,0]) ) /  \
        (c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) + v**2*c2ptM[tf,1,1] )[None]    \
        for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)]
    tf2ratio_GEVP_w[tf]=np.real(np.array(t))
    
tf2ratio_GEVP_0w={}
for tf in tfs:
    t=[ (c3ptM[:,0,0]*(1) + v*(1)*(c3ptM[:,0,1]+c3ptM[:,1,0]) ) /  \
        (c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) + v**2*c2ptM[tf,1,1] )[None]    \
        for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)]
    tf2ratio_GEVP_0w[tf]=np.real(np.array(t))
    
tf2ratio_GEVP_0w_0v2={}
for tf in tfs:
    t=[ (c3ptM[:,0,0]*(1) + v*(1)*(c3ptM[:,0,1]+c3ptM[:,1,0]) ) /  \
        (c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) )[None]    \
        for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)]
    tf2ratio_GEVP_0w_0v2[tf]=np.real(np.array(t))
    
dic={'tf2ratio':tf2ratio_GEVP_w, 'fillstyle:[global]':['full'], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'])
dic={'tf2ratio':tf2ratio_GEVP_0w, 'fillstyle:[global]':['left'], 'shift:[rainbow,midpoint,fit]':[-0.3,-0.4,0], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'],figAxs=(fig,axs))
dic={'tf2ratio':tf2ratio_GEVP_0w_0v2, 'fillstyle:[global]':['right'], 'shift:[rainbow,midpoint,fit]':[0.3,0.4,0], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'],figAxs=(fig,axs))
# fig.suptitle('0w vs w vs w_0v2')

ax=axs[0,0]
ax.set_ylabel(r'$\sigma_{\pi N}$ [MeV]')
ax.set_ylim([0,80])
ax.set_yticks([0,20,40,60,80])
ax.set_xlim([-0.8,0.8])
ax.set_xticks(np.arange(-0.6,0.7,0.2))
handles=[plt.errorbar(np.nan,np.nan,yerr=np.nan,color='black',fmt='s',mfc=None,fillstyle=fillstyle) for fillstyle in ['left','full','right']]
yu.legend(ax,labels=[r'$R_{\rm GEVP}^{\prime}$',r'$R_{\rm GEVP}$',r'$R_{\rm GEVP}^{\prime\prime}$'],ncols=3,tightQ=True,handles=handles,fontsize=20)

ax=axs[0,1]
ax.set_xticks(np.arange(0.7,1.6,0.3))

yu.finalizePlot('Rd_compare_w_v2')

In [8]:
n=2
def dE2tf2ratio(dE):
    lbd0=dE*n
    lbd=np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    
    tf2ratio={}
    for tf in tfs:
        c2=np.array([c2ptM[tf,0,0]   for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        c3=np.array([c3ptM[:,0,0] for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        
        c3pt=-(np.roll(c3,-n,axis=-1)+np.roll(c3,n,axis=-1)-2*c3) + lbd[:,None]**2*c3
        c2pt=(lbd**2)*c2
        
        ratio=c3pt/c2pt[:,None]
        tf2ratio[tf]=ratio
    return tf2ratio

def dE2tf2ratio_GEVP(dE):
    lbd0=dE*n
    lbd=np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    
    tf2ratio={}
    for tf in tfs:
        c2=np.array([c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) + v**2*c2ptM[tf,1,1]  for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        c3=np.array([c3ptM[:,0,0]*(1-w**2) + v*(1+w)*(c3ptM[:,0,1]+c3ptM[:,1,0])  for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        
        c3pt=-(np.roll(c3,-n,axis=-1)+np.roll(c3,n,axis=-1)-2*c3) + lbd[:,None]**2*c3
        c2pt=(lbd**2)*c2
        
        ratio=c3pt/c2pt[:,None]
        tf2ratio[tf]=ratio
    return tf2ratio

def dEdE2tf2ratio_GEVP(dE1,dE2,n1=n,n2=n):
    lbd0=dE1*n1; lbd1=np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    lbd0=dE2*n2; lbd2=np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    
    tf2ratio={}
    for tf in tfs:
        c2=np.array([c2pt + v*(c2ptM[tf,0,1]+c2ptM[tf,1,0]) + v**2*c2ptM[tf,1,1]  for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        c3=np.array([c3ptM[:,0,0]*(1-w**2) + v*(1+w)*(c3ptM[:,0,1]+c3ptM[:,1,0])  for c3ptM,c2ptM,c2pt,v,w in zip(tf2c3ptM[tf],c2ptM,c2ptCorrDic_NJN[tf],v_selected,w_selected)])
        
        c3=-(np.roll(c3,-n1,axis=-1)+np.roll(c3,n1,axis=-1)-2*c3) + lbd1[:,None]**2*c3
        c3pt=-(np.roll(c3,-n2,axis=-1)+np.roll(c3,n2,axis=-1)-2*c3) + lbd2[:,None]**2*c3
        
        c2pt=(lbd1**2)*(lbd2**2)*c2
        
        ratio=c3pt/c2pt[:,None]
        tf2ratio[tf]=ratio
    return tf2ratio

In [9]:
fits=yu.getFits(f'gS+_{ens}_lbd_True',pathlabel='analysis_3pt_laplace')
fit=[fit for fit in fits if fit[0]==(8,4)][0]
dE=fit[1][:,1]

In [10]:
dic={'tf2ratio':tf2ratio_noGEVP, 'mfc:[global]':['white'], 'shift:[rainbow,midpoint,fit]':[-0.2,-0.2,0], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'])
axs[0,0].set_ylim([0,80])
dic={'tf2ratio':dE2tf2ratio(dE), 'rainbow:[tfmin,tfmax,tcmin,dt]': [None,None,3,None], 'mfc:[global]':['None'], 'xyunit':(yu.ens2a[ens],yunit_mul) }
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow','midpoint'],figAxs=(fig,axs))
# fig.suptitle('open (R_std); filled (R_lap)')

ax=axs[0,0]
ax.set_ylabel(r'$\sigma_{\pi N}$ [MeV]')
ax.set_ylim([0,80])
ax.set_yticks([0,20,40,60,80])
ax.set_xlim([-0.8,0.8])
ax.set_xticks(np.arange(-0.6,0.7,0.2))
handles=[plt.errorbar(np.nan,np.nan,yerr=np.nan,color=color,fmt=fmt,mfc=mfc) for color,fmt,mfc in zip(['black','black'],['s','s'],['white',None])]
yu.legend(ax,labels=[r'$R_{\rm std}$',r'$R_{\rm Lap}$'],ncols=2,tightQ=True,handles=handles,fontsize=20)

ax=axs[0,1]
ax.set_xticks(np.arange(0.7,1.6,0.3))

yu.finalizePlot('Rstd_Rlap')